# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from ollama import Client

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OLLAMA_API_KEY')

if api_key and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = os.getenv('MODEL_NAME')

API key looks good so far


In [3]:
client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
)

In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [20]:
def select_relevant_links(url):
    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        format={"type": "json_object"}
    )
    result = response.message.content
    links = json.loads(result)
    return links
    

In [21]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'product page – Proficient',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'product page – Connect Four',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'product page – Outsmart',
   'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'curriculum / course offerings',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog / posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social media – LinkedIn',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media – Twitter',
   'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media – Facebook',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'partner platform – Nebula.io', 'url': 'https://nebula.io/'}]}

In [36]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        format='json'
    )
    result = response.message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [37]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-oss:20b-cloud
Found 2 relevant links


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [38]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b-cloud
Found 8 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'career application',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'API endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'learn resources', 'url': 'https://huggingface.co/learn'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [27]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [28]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b-cloud
Found 1 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-4.7-Flash
Updated
5 days ago
•
363k
•
1.16k
nvidia/personaplex-7b-v1
Updated
3 days ago
•
29.4k
•
912
microsoft/VibeVoice-ASR
Updated
4 days ago
•
21.7k
•
495
Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice
Updated
3 days ago
•
42.7k
•
412
unsloth/GLM-4.7-Flash-GGUF
Updated
1 day ago
•
196k
•
319
Browse 2M+ models
Spaces
Running
on
Zero
627
Qwen3-TTS Demo
🎙
627
Convert text to speech with custom voices and cloning
Running
on
Zero
Featured
1.14k
Qwen Image Multiple Angles 3D Camera
🎥
1.14k
Adjust camera angles in images using 3

In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [30]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [32]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b-cloud
Found 4 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-4.7-Flash\nUpdated\n5 days ago\n•\n363k\n•\n1.16k\nnvidia/personaplex-7b-v1\nUpdated\n3 days ago\n•\n29.4k\n•\n912\nmicrosoft/VibeVoice-ASR\nUpdated\n4 days ago\n•\n21.7k\n•\n495\nQwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\nUpdated\n3 days ago\n•\n42.7k\n•\n412\nunsloth/GLM-4.7-Flash-GGUF\nUpdated\n1 day ago\n•\n196k\n•\n319\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\n627\nQwen3-TT

In [40]:
def create_brochure(company_name, url):
    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.message.content
    display(Markdown(result))

In [41]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b-cloud
Found 9 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face  
**The AI community building the future**

---

## Who We Are  
Hugging Face is the collaborative hub that brings together researchers, engineers, artists, and hobbyists to create, share, and accelerate machine‑learning innovation. With an open‑source philosophy and a vibrant community, we empower anyone to build intelligent applications that span text, images, audio, video, and even 3‑D content.

### Core Mission  
- **Open collaboration:** 2 M+ models, 500 k+ datasets, and 1 M+ live applications are hosted, discovered, and evolved by the community.  
- **Democratize AI:** Easy‑to‑use web interface, “Spaces” for instant demos, and a starter‑kit that lets you go from notebook to production in minutes.  
- **Scalable enterprise solutions** with security, single‑sign‑on, and customizable support.

---

## What We Offer  

| Feature | What It Means | Value |
|---------|---------------|-------|
| **Models** | Pre‑trained, fine‑tuned, or custom AI models for any task | 2 M+ public models, thousands of active downloads |
| **Datasets** | Curated data for training, evaluation, and research | 500 k+ datasets across text, vision, audio, and 3‑D |
| **Spaces** | Zero‑cost, on‑prem or cloud‑hosted demos and apps | Run Vecjs, text‑to‑speech, image editing on command |
| **Docs & SDK** | Extensive references and community tutorials | Boost model‑interpretability and experimentation |
| **Enterprise Hub** | Team plans from $20/user/month, Enterprise contracts | SSO, dedicated support, compliance controls, regional hosting |
| **Community** | Forums, Slack, and collaboration meetings | A global network of developers and researchers |

---

## Community Highlights  
- **Trending Models:** GLM‑4.7‑Flash, Qwen‑TTS 12Hz, FLUX.2 9B, and many more, updated daily.  
- **Live Demo Library:** 631 running demos such as Qwen3‑TTS and Z Image Turbo.  
- **Modality Coverage:** Seamlessly handle **text, image, video, audio, and 3‑D** from a single platform.  
- **Portfolio Builder:** Share your models & datasets, earn recognition, and build a professional profile visible to recruiters.

---

## Serving Industries & Customers  
Hugging Face’s platform powers solutions for:

| Sector | Use Cases |
|-------|-----------|
| **Automotive** | Voice‑controlled infotainment, autonomous‑driving perception |
| **Healthcare** | Medical record NLP, medical imaging |
| **Finance** | Fraud detection, sentiment analysis, portfolio optimization |
| **Media & Entertainment** | Generative art, text‑to‑speech, content moderation |
| **Retail & E‑commerce** | Recommendation engines, customer support bots |
| **Education** | Adaptive learning systems, tutoring assistants |

Our enterprise tier offers **security‑first contracts, managed hosting, and SSO** for enterprises of all sizes—from small startups to Fortune 500 companies.

---

## Careers at Hugging Face  
> **Hugging Face – Current Openings**  
> Join a rapidly growing, mission‑driven team.

We value:
- **Curiosity** – a love for learning and exploring new ideas.
- **Collaboration** – open to cross‑disciplinary teamwork.
- **Impact** – a desire to build technology that improves the world.

Roles span **engineer, data scientist, product manager, community advocate, and more**. Explore the latest openings on our careers portal and help shape the next era of AI.

---

## Get Involved  

| Action | How |
|---------|-----|
| **Explore models** | Visit the Models hub and discover 2 M+ AI assets |
| **Build an app** | Deploy a Hugging Face Space in minutes |
| **Contribute** | Fork, modify, and share your own dataset or model |
| **Join Enterprise** | Sign up for the Team plan or contact sales for custom Enterprise contracts |
| **Apply to work** | Check the careers page and submit your application |

---

### Connect with Us  
- **Website:** [huggingface.co](https://huggingface.co)  
- **Community:** Discord, Slack, GitHub, Twitter, and more  
- **Enterprise Contacts:** sales@huggingface.com  

Let’s build the future of AI together—empowering creativity, fostering collaboration, and delivering cutting‑edge technology to every corner of the world.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [47]:
def stream_brochure(company_name, url):
    stream = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.message.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [48]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b-cloud
Found 15 relevant links


# Hugging Face – The AI Community Building the Future  

**Where the world of machine learning converges, collaborates, and thrives.**

---

## 🚀 What We Are

- **An open‑source, community‑first platform** that lets anyone create, share, and discover AI models, datasets, and applications.  
- Hosts **2 M+ publicly‑available models** and **500 k+ datasets** across every modality—text, image, video, audio, and even 3‑D.  
- Offers **Spaces** – instant‑run, interactive demos that let you experiment with state‑of‑the‑art models with a single click.  
- Built on the **HF Open‑Source stack**, giving developers a lightweight, performant foundation to build the next generation of AI.  

---

## 🌐 The Hugging Face Ecosystem

| Area | Highlights |
|------|------------|
| **Models** | 2 M+ models, from LLMs like *GLM‑4.7‑Flash* to audio and vision. |
| **Datasets** | 500 k+ curated datasets, from *Alibaba‑Apsara/Superior‑Reasoning‑SFT* to *Facebook/action100m-preview*. |
| **Spaces** | 1 M+ user‑created AI apps—ready to run on “Zero” (instant inference) or be deployed at scale. |
| **Docs** | Complete APIs, tutorials, and reference—designed for rapid onboarding. |
| **Community** | A vibrant network of researchers, developers, and entrepreneurs who review, improve, and extend every model. |

---

## 🏢 Enterprise Solutions  

| Feature | Value |
|---------|-------|
| **Security & Compliance** | Enterprise‑grade access controls, single sign‑on (SSO) integration, region‑specific data residency, and comprehensive audit logs. |
| **Pricing** | **Team** starts at **$20/user/month**; **Enterprise** offers flexible contracts tailored to your organization’s needs. |
| **Scalability** | Build teams, collaborate safely, and deploy production‑ready AI across your organization with confidence. |
| **Support** | Dedicated support and governance tools for a smoother, faster rollout. |

---

## 🎯 Who Uses Hugging Face

- **Researchers** looking to reproduce state‑of‑the‑art results.  
- **Developers & ML Engineers** building production systems that require reliable, open‑source models.  
- **Product Managers & Innovators** who want to prototype quickly with the latest LLMs, vision, or audio models.  
- **Enterprises** that need scalable, secure AI infrastructure—from fintech to healthcare to media.  

---

## 🤝 Culture & Values

| Pillar | How We Embody It |
|--------|-----------------|
| **Open Source** | Every model, dataset, and space is freely available and continuously improved by the community. |
| **Collaboration** | By “hosting and collaborating on unlimited public assets”, we encourage knowledge sharing and collective progress. |
| **Innovation** | From 3‑D camera control image tools to real‑time text‑to‑speech clones, our platform bridges cutting‑edge research and practice. |
| **Inclusivity** | A global community welcoming all skill levels—from hobbyists to industry leaders. |

---

## 👩‍💻 Careers & Growth

> *“Join a team that’s reshaping AI for the public good.”*  

- **Open roles—from research to engineering, product to community management.**  
- Work in a fast‑moving, open‑source environment that directly impacts millions of developers worldwide.  
- **Remote‑first** values and a culture that prizes learning, experimentation, and shared success.

> **[Explore opportunities → Careers page](https://huggingface.co/careers)**  

---

## 📣 Join the Revolution

- **Sign Up** for free, start sharing or modeling, and become part of the **AI community building the future**.  
- **Developer**: Dive into the **HF Open‑Source stack** today.  
- **Enterprise**: Contact **sales** to scale your organization with Hugging Face.  

**Explore, Build, Collaborate – All on a Single, Powered‑by‑Community Platform.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

# stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>